In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import cv2
# Replace YOUR_USERNAME with ur kaggle username
RAW_DIR              = r"/kaggle/input/datasets/YOUR_USERNAME/raw-data" # Main folder of CSV files
MIMIC_SPLIT_CSV      = RAW_DIR + r"/mimic-cxr-2.0.0-split.csv"
MIMIC_LABELS_CSV     = RAW_DIR + r"/mimic-cxr-2.0.0-chexpert.csv"
MIMIC_PATIENTS_CSV   = RAW_DIR + r"/patients.csv"
MIMIC_ADMISSIONS_CSV = RAW_DIR + r"/admissions.csv"

OUTPUT_DIR     = Path(r"/kaggle/working/")
os.makedirs(OUTPUT_DIR, exist_ok=True)

P_BATCH_PREFIX = [10, 11]

BATCH_ROOTS = {
    "p10": Path(r"/kaggle/input/datasets/YOUR_USERNAME/p10-batch"), # Main folder of p10 and p11 batch images (Retrieve from MIMIC-CXR)
    "p11": Path(r"/kaggle/input/datasets/YOUR_USERNAME/p11-batch"),
}

In [ ]:
# Load split csv
meta_df = pd.read_csv(MIMIC_SPLIT_CSV)
print(f"Metadata CSV    : {meta_df.shape}")
print(meta_df.head(3))

# Load labels csv
labels_df = pd.read_csv(MIMIC_LABELS_CSV)
print(f"Labels CSV      : {labels_df.shape}")
print(labels_df.head(3))

# Load patients and admissions csv
patients_df   = pd.read_csv(MIMIC_PATIENTS_CSV)
admissions_df = pd.read_csv(MIMIC_ADMISSIONS_CSV)
print(f"Patients CSV    : {patients_df.shape}")
print(f"Admissions CSV  : {admissions_df.shape}")

In [ ]:
# Merge labels and demographics
df = meta_df.copy()
if labels_df is not None:
    df = df.merge(
        labels_df[['subject_id', 'study_id', 'Cardiomegaly']],
        on=['subject_id', 'study_id'],
        how='left'
    )
    print("Merged label columns.")

if patients_df is not None:
    df = df.merge(
        patients_df[['subject_id', 'gender', 'anchor_age']],
        on='subject_id',
        how='left'
    )
    print("Merged patient demographics.")

if admissions_df is not None:
    # Use the first admission per patient for race
    race_df = (
        admissions_df[['subject_id', 'race']]
        .drop_duplicates(subset='subject_id', keep='first')
    )
    df = df.merge(race_df, on='subject_id', how='left')
    print("Merged race/ethnicity.")

print(f"\nMerged dataframe shape: {df.shape}")
print(df.head(3))

In [ ]:
# Standardise and clean columns
# CheXpert: 1.0 = positive, 0.0 = negative, -1.0 = uncertain, NaN = not mentioned
# Strategy: treat uncertain (-1) as positive (conservative, avoids false negatives)
df['Cardiomegaly'] = df['Cardiomegaly'].replace(-1.0, 0.0)   # Uncertain → Negative
df['Cardiomegaly'] = df['Cardiomegaly'].fillna(0.0)          # Not mentioned → Negative
df['Cardiomegaly'] = df['Cardiomegaly'].astype(int)

# Alternative: drop uncertain labels
# df = df[df['Cardiomegaly'] != -1.0].copy()
# df['Cardiomegaly'] = df['Cardiomegaly'].fillna(0.0).astype(int)
print("Cardiomegaly value counts:")
print(df['Cardiomegaly'].value_counts())

In [ ]:
# Standardise race labels
def standardise_race(race_str):
    if pd.isna(race_str):
        return 'Unknown'
    r = str(race_str).upper()
    if 'WHITE' in r:
        return 'White'
    elif 'BLACK' in r or 'AFRICAN' in r:
        return 'Black'
    elif 'ASIAN' in r:
        return 'Asian'
    elif 'HISPANIC' in r or 'LATINO' in r:
        return 'Hispanic'
    elif 'UNKNOWN' in r or 'UNABLE' in r or 'DECLINED' in r:
        return 'Unknown'
    else:
        return 'Other'

if 'race' in df.columns:
    df['race'] = df['race'].apply(standardise_race)
    print("Race value counts after standardisation:")
    print(df['race'].value_counts())
else:
    df['race'] = 'Unknown'
    print("No 'race' column found, filled with 'Unknown'.")

# Standardise gender labels
if 'gender' in df.columns:
    df['gender'] = df['gender'].str.strip().str.capitalize().fillna('Unknown')
    print("Gender value counts:")
    print(df['gender'].value_counts())
else:
    df['gender'] = 'Unknown'
    print("No 'gender' column found, filled with 'Unknown'.")

In [ ]:
# Checking if required columns are existed
REQUIRED_COLS = ['subject_id', 'study_id', 'dicom_id', 'split', 'Cardiomegaly', 'race', 'gender']
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
else:
    print("All required columns present.")

# Keep only required columns (plus age if available)
keep_cols = REQUIRED_COLS + (['anchor_age'] if 'anchor_age' in df.columns else [])
df = df[keep_cols]

In [ ]:
# Filter csv files to p10-p11 batches
if P_BATCH_PREFIX is not None:
    before = len(df)
    prefixes = tuple(map(str, P_BATCH_PREFIX))  # convert list → tuple of strings
    df = df[df['subject_id'].astype(str).str.startswith(prefixes)].reset_index(drop=True)
    print(f"Filtered to p{P_BATCH_PREFIX} batch: {before} → {len(df)} records")
else:
    pass

In [ ]:
# Drop NA and validate
before = len(df)
df = df.dropna(subset=['subject_id', 'study_id', 'dicom_id', 'Cardiomegaly']).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with null required fields. Remaining: {len(df)}")

# Validate split column
print("\nSplit distribution:")
print(df['split'].value_counts())

# Validate label balance per split
print("\nCardiomegaly prevalence per split:")
print(df.groupby('split')['Cardiomegaly'].mean().round(3))

In [ ]:
# Create and save split
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'validate'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

print(f"Train : {len(train_df):>5} rows  |  Cardiomegaly+: {train_df['Cardiomegaly'].sum()}")
print(f"Val   : {len(val_df):>5} rows  |  Cardiomegaly+: {val_df['Cardiomegaly'].sum()}")
print(f"Test  : {len(test_df):>5} rows  |  Cardiomegaly+: {test_df['Cardiomegaly'].sum()}")

# Save
train_df.to_csv(os.path.join(OUTPUT_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, 'val.csv'),   index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, 'test.csv'),  index=False)
df.to_csv(os.path.join(OUTPUT_DIR, 'full.csv'),       index=False)

print(f"\n All splits saved to: {OUTPUT_DIR}")

In [ ]:
# JPEG Reading
def get_jpg_path(subject_id, study_id, dicom_id, batch_roots=BATCH_ROOTS):
    prefix = f"p{str(subject_id)[:2]}"
    base   = batch_roots.get(prefix, list(batch_roots.values())[0])
    return base / prefix / prefix / f"p{subject_id}" / f"s{study_id}" / f"{dicom_id}.jpg"

sums, sums_squared = 0, 0

for c, subject_id in enumerate(tqdm(df['subject_id'])):
    subject_id = df['subject_id'].iloc[c]
    study_id   = df['study_id'].iloc[c]
    dicom_id   = df['dicom_id'].iloc[c]

    jpg_path = get_jpg_path(subject_id, study_id, dicom_id)

    jpg       = np.array(Image.open(jpg_path), dtype=np.float32) / 255.0
    jpg_array = cv2.resize(jpg, (224, 224)).astype(np.float16)

    label      = df['Cardiomegaly'].iloc[c]
    split_val  = df['split'].iloc[c]

    train_val_test = {"train": "train", "validate": "val"}.get(split_val, "test")

    current_save_path = OUTPUT_DIR / train_val_test / str(label)
    current_save_path.mkdir(parents=True, exist_ok=True)
    np.save(current_save_path / f"{dicom_id}.npy", jpg_array)

    normalizer = 224 * 224
    if train_val_test == "train":
        sums         += np.sum(jpg_array) / normalizer
        sums_squared += (jpg_array ** 2).sum() / normalizer

In [ ]:
# Retrieve mean and std values (for part 2 of the notebook)
mean = sums / 24000
std = np.sqrt(sums_squared / 24000 - (mean**2))